In [ ]:
import json
import sys
from pprint import pprint

sys.stdin.reconfigure(encoding="utf-8") # ???

import pandas as pd
from openai import OpenAI
from openai.types.chat import ChatCompletionMessage

import os
from dotenv import load_dotenv

load_dotenv()

OPENROUTER_API_KEY  = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_URL      = "https://openrouter.ai/api/v1"

ELICE_API_KEY       = os.getenv("ELICE_API_KEY")
ELICE_URL           = os.getenv("ELICE_URL")

# MODEL_NAME          = 'nvidia/nemotron-3-super-120b-a12b:free'
MODEL_NAME          = "zai-org/glm-4.7"

# client = OpenAI(
#     api_key  = OPENROUTER_API_KEY, # 여기에 실제 API 키를 넣으세요
#     base_url = OPENROUTER_URL,
# )

client = OpenAI(
    api_key  = ELICE_API_KEY, # 여기에 실제 API 키를 넣으세요
    base_url = ELICE_URL,
)

def find_products(keywords: list[str]) -> list[dict]:
    """제품 설명에 keywords가 포함된 제품을 찾아서 반환합니다."""

    print("find_products 함수", keywords)
    df      = pd.read_csv("product_list.csv")
    result  = df[df["description"].str.contains("|".join(keywords))]
    return result.to_dict(orient="records")


def get_response(chat_log: list, tools: dict = None) -> ChatCompletionMessage:
    response = client.chat.completions.create(
        model    = MODEL_NAME,
        messages = chat_log,
        tools    = tools
    )

    if response.choices[0].message.tool_calls:
        tool_call_results = []
        for tool_call in response.choices[0].message.tool_calls:
            call_id = tool_call.id

            if tool_call.function.name == "find_products":
                keywords = json.loads(tool_call.function.arguments)["keywords"]
                products = find_products(keywords)

                call_result_message = {
                    "tool_call_id"  : call_id,
                    "role"          : "tool",
                    "content"       : json.dumps(
                        {"keywords": keywords, "products": str(products)},
                        ensure_ascii = False,
                    ),
                }

                tool_call_results.append(call_result_message)

        messages = chat_log + [response.choices[0].message] + tool_call_results
        response = client.chat.completions.create(
            model    = MODEL_NAME,
            messages = messages,
            tools    = tools,
        )

    return response.choices[0].message


find_products_info = {
    "name": "find_products",
    "description": "제품 설명에 keywords가 포함된 제품을 찾아서 반환합니다. 찾고자 하는 keywords는 사용자가 요청한 내용과 관련된 제품을 필터링할 수 있는 적절한 키워드여야 합니다.",
    "parameters": {
        "type": "object",
        "properties": {
            "keywords": {
                "type": "array",
                "items": {"type": "string"},
                "description": "검색할 키워드 리스트. 사용자의 발화에서 추출한 핵심 명사/혀용사를 1~5개 사이로 전달한다.",  # 1. description을 적절한 내용으로 채우세요.
            },
        },
        "required": ["keywords"],
        "additionalProperties": False,
    },
}

tools = [{"type": "function", "function": find_products_info}]


def main():
    chat_log = [
        {
            "role": "system",
            "content": "너는 쇼핑마트의 AI 챗봇이야. 다양한 제품에 대해 물어보거나 추천을 하는 용도야. 추천을 할 때는 상품 목록에서 사용자가 원하는 상품만 선별해서 추천해야 해.",
        },
    ]

    # 2. chat_log에 사용자가 입력한 대화를 추가하세요.
    chat_log.append(
        {
            "role": "user", 
            "content": input("You>")
        }
    )

    response = get_response(chat_log, tools = tools)
    print("AI>", response.content)


if __name__ == "__main__":
    main()


find_products 함수 ['다이어트', '식단', '저칼로리', '건강', '다이어트식']
AI> 다이어트 식단으로 추천드릴 만한 제품들을 찾아봤어요! 🥗

## 📋 다이어트 식단 추천 목록

### 🥗 샐러드류
| 제품 | 가격 | 설명 | 재고 |
|------|------|------|------|
| **다이어트 샐러드A** | 7,000원 | 견과류 포함 완제품 샐러드 | 10개 |
| **다이어트 샐러드B** | 9,000원 | 연어 추가 프리미엄 샐러드 | 5개 |
| **아보카도 샐러드** | 8,500원 | 아보카도와 신선한 채소 | 7개 |
| **퀴노아 샐러드** | 9,000원 | 퀴노아와 야채가 가득 | 5개 |

### 🥩 고단백 메인
| 제품 | 가격 | 설명 | 재고 |
|------|------|------|------|
| **닭가슴살 스낵** | 4,000원 | 고단백 저지방, 50g | 20개 |
| **저칼로리 치킨 샌드위치** | 7,000원 | 가볍게 드시기 좋은 샌드위치 | 8개 |
| **매콤한 비건 버거** | 7,800원 | 채식주의자용 비건 버거 | 8개 |

### 🥤 단백질 보충/음료
| 제품 | 가격 | 설명 | 재고 |
|------|------|------|------|
| **바나나 프로틴쉐이크** | 5,500원 | 350ml 단백질 보충 음료 | 10개 |
| **코코아 프로틴 쉐이크** | 6,000원 | 350ml 코코아 맛 | 12개 |

### 🍫 간식
| 제품 | 가격 | 설명 | 재고 |
|------|------|------|------|
| **초코 프로틴바** | 3,500원 | 50g 고단백 초콜릿 바 | 15개 |
| **카카오닙스 스낵** | 3,500원 | 건강한 다이어트 스낵 | 20개 |

---

⚠️ **알레르기 주의**
- 견과류(아몬드 등) 알러지 고객: 다이어트샐러드A, 초코 프로틴바, 아보카도샐러드, 치즈그래놀라바, 비건버거 주의
- 유당 불내증: 바나